# 01 — Exploratory data analysis (CirCor DigiScope, public part)

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, soundfile as sf, librosa, librosa.display
from src.data import load_patients, load_recordings, load_split, WAV_DIR
from src.metrics import weighted_accuracy
pd.set_option('display.width', 140)

## Patients and labels

In [ ]:
pat = load_patients()
print(len(pat), 'patients')
pat['Murmur'].value_counts().to_frame().assign(pct=lambda d: (100*d['count']/d['count'].sum()).round(1))

In [ ]:
# Baseline: always predicting one class
for c in ['Absent', 'Present', 'Unknown']:
    print(f'always {c:8s} WA = {weighted_accuracy(pat.Murmur, [c]*len(pat)):.3f}')

In [ ]:
pd.crosstab(pat['Age'].fillna('NA'), pat['Murmur'])

## Recordings

In [ ]:
rec = load_recordings()
info = [sf.info(p) for p in rec.path]
rec['sr'] = [i.samplerate for i in info]
rec['duration'] = [i.duration for i in info]
print(len(rec), 'recordings;', rec.patient_id.nunique(), 'patients; sample rates:', rec.sr.unique())
rec.groupby('patient_id').size().value_counts().sort_index().rename('patients with n recordings')

In [ ]:
print(rec.duration.describe().round(1))
rec.location.value_counts()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
rec.duration.hist(bins=60, ax=ax[0]); ax[0].set(xlabel='duration (s)', ylabel='recordings')
rec.groupby('murmur').duration.plot.kde(ax=ax[1], legend=True); ax[1].set(xlabel='duration (s)', xlim=(0, 60))
plt.tight_layout()

## Murmur locations
For *Present* patients the murmur may be audible only at some locations: recording-level labels inherited from
the patient label are noisy. `Murmur locations` lets us build cleaner recording-level labels for training.

In [ ]:
present = pat[pat.Murmur == 'Present']
n_rec = present['Recording locations:'].str.split('+').apply(lambda l: len(set(l)))
n_mur = present['Murmur locations'].str.split('+').apply(len)
pd.DataFrame({'recorded locations': n_rec, 'murmur locations': n_mur}).value_counts().sort_index()

## Segmentation files (.tsv)
Each recording comes with a segmentation: `start, end, state` (0 = unannotated, 1 = S1, 2 = systole, 3 = S2, 4 = diastole).

In [ ]:
tsv = pd.read_csv(WAV_DIR / (rec.recording.iloc[0] + '.tsv'), sep='\t', header=None, names=['start', 'end', 'state'])
tsv.head(10)

## Example signals

In [ ]:
ex = {m: rec[rec.murmur == m].iloc[0] for m in ['Absent', 'Present', 'Unknown']}
fig, axes = plt.subplots(3, 2, figsize=(13, 8))
for (m, r), (a1, a2) in zip(ex.items(), axes):
    y, sr = librosa.load(r.path, sr=None, duration=5)
    a1.plot(np.arange(len(y)) / sr, y, lw=.5); a1.set_title(f'{m} — {r.recording}')
    S = librosa.amplitude_to_db(np.abs(librosa.stft(y, n_fft=512, hop_length=64)), ref=np.max)
    librosa.display.specshow(S, sr=sr, hop_length=64, x_axis='time', y_axis='linear', ax=a2); a2.set_ylim(0, 1000)
plt.tight_layout()

## Split check

In [ ]:
split = load_split()
pd.crosstab(split.fold, split.murmur, margins=True)